In [51]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read() # utf-8编码，读取input.txt文件内容

In [3]:
print("Length of text:", len(text)) # 输出文本长度

Length of text: 1115394


In [8]:
chars = sorted(list(set(text))) # 获取文本中所有不同的字符，并排序
vocab_size = len(chars) # 计算不同字符的数量
print(''.join(chars)) # 输出所有不同的字符
print("Vocab size:", vocab_size) # 输出不同字符的数量


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


In [9]:
stoi = { ch: i for i, ch in enumerate(chars) } # 创建字符到索引的映射
itos = { i: ch for i, ch in enumerate(chars) } # 创建索引到字符的映射
encode = lambda s: [stoi[c] for c in s] # 定义编码函数，将字符串转换为索引列表

# 为什么decode要用''.join()呢？
# 因为decode函数的目的是将索引列表转换回字符串，而索引列表中的每个元素都是一个字符的索引。
# 使用''.join()可以将这些字符连接成一个完整的字符串。
# 如果不使用''.join()，则会得到一个包含单个字符的列表，而不是一个字符串。
# 因此，''.join()是必要的，以确保最终结果是一个字符串而不是列表。
decode = lambda l: ''.join([itos[i] for i in l]) # 定义解码函数，将索引列表转换为字符串
print(encode("hello world")) # 测试编码函数
print(decode(encode("hello world"))) # 测试解码函数

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [11]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)  # 将编码后的文本转换为PyTorch张量，数据类型为长整型
print(data.shape, data.dtype) # 输出张量的形状和数据类型
print(data[:100]) # 输出张量的前100个元素

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [12]:
# 划分训练集和验证集
n = int(0.9*len(data))
train_data = data[:n] # 训练集
val_data = data[n:] # 验证集

In [13]:
block_size = 8 # 每个输入序列的长度
train_data[:block_size+1] # 输出训练集的前9个元素（8个输入 + 1个目标）

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size] # 输入序列
y = train_data[1:block_size+1] # 目标序列
for t in range(block_size):
    context = x[:t+1] # 上下文序列，长度为t+1
    target = y[t] # 目标字符
    print(f"when input is {context.tolist()} the target: {target}") # 输出上下文序列和目标字符

when input is [18] the target: 47
when input is [18, 47] the target: 56
when input is [18, 47, 56] the target: 57
when input is [18, 47, 56, 57] the target: 58
when input is [18, 47, 56, 57, 58] the target: 1
when input is [18, 47, 56, 57, 58, 1] the target: 15
when input is [18, 47, 56, 57, 58, 1, 15] the target: 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target: 58


In [17]:
torch.manual_seed(1337) # 设置随机种子，确保结果可复现
batch_size = 4 # 每个批次的样本数量
block_size = 8 # 每个输入序列的长度
def get_batch(split):
    data = train_data if split == 'train_data' else val_data # 根据split参数选择训练集或验证集
    # 為什么要用len(data) - block_size呢？
    # 因为我们需要从数据中提取长度为block_size的输入序列和长度为block_size的目标序列。
    # 如果我们直接使用len(data)，那么在选择起始索引时，可能会超出数据范围。
    # 因此，我们需要确保起始索引加上block_size不会超过数据的长度。
    # (batch_size,)表示生成一个形状为(batch_size,)的一维张量，里面的元素是从0到len(data) - block_size - 1之间的随机整数。
    ix = torch.randint(len(data) - block_size, (batch_size,)) # 随机选择batch_size个起始索引
    # torch.stack()函数的作用是将一系列张量沿着一个新的维度进行拼接，形成一个新的张量。
    x = torch.stack([data[i:i+block_size] for i in ix]) # 构建输入序列
    y = torch.stack([data[i+1:i+1+block_size] for i in ix]) # 构建目标序列
    return x, y # 返回输入序列和目标序列

xb, yb = get_batch('train_data') # 获取一个训练批次
print('inputs:')
print(xb.shape) # 输出输入序列的形状
print(xb) # 输出输入序列
print('targets:')
print(yb.shape) # 输出目标序列的形状
print(yb) # 输出目标序列

print('----')

for b in range(batch_size): # b表示批次中的第b个样本
    for t in range(block_size): # t表示时间步，即第t个时间步的输入和目标
        context = xb[b, :t+1] # 获取第b个样本的前t+1个时间步作为上下文
        target = yb[b, t] # 目标字符
        print(f"when input is {context.tolist()} the target: {target}") # 输出上下文序列和目标字符

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [18]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337) # 设置随机种子，确保结果可复现

class BigramLanguageModel(nn.Module): # 定义一个大ram语言模型类，继承自nn.Module
    def __init__(self, vocab_size):
        super().__init__()  # 调用父类的构造函数
        # 定义一个嵌入层，将词汇表大小映射到词汇表大小的向量空间
        # 这里为什么要用nn.Embedding(vocab_size, vocab_size)呢？
        # 因为我们希望将每个词汇表中的索引映射到一个大小为vocab_size的向量空间中，这样每个索引都会有一个对应的向量表示。
        # nn.Embedding的第一个参数是词汇表的大小，第二个参数是嵌入向量的维度。
        # 在这个例子中，我们将嵌入向量的维度设置为vocab_size，这样每个索引都会有一个大小为vocab_size的向量表示。
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) 

    def forward(self, idx, targets):

        # 将输入的tokens映射到logits
        # logits的形状是 (batch_size, block_size, vocab_size),怎么构造成这样的?
        # 1. idx的形状是 (batch_size, block_size)
        # 2. self.token_embedding_table(idx)会将每个token的索引映射到一个大小为vocab_size的向量,
        # 所以输出的形状是 (batch_size, block_size, vocab_size)
        logits = self.token_embedding_table(idx)
        if targets is None: # 如果没有提供目标tokens，则直接返回logits
            loss = None
        else:
            B, T, C = logits.shape # B是batch_size，T是block_size，C是vocab_size
            # 为什么要将logits展平为 (B*T, C)的形状呢？
            # 因为在计算交叉熵损失时，PyTorch的F.cross_entropy函数要求输入的logits的形状是 (N, C)，
            # 其中N是样本的数量，C是类别的数量。在我们的例子中，N应该是B*T，因为我们有B个样本，每个样本有T个时间步。
            # 同样地，targets也需要展平为 (B*T,)的形状
            logits = logits.view(B*T, C) # 将logits展平为 (B*T, C)的形状，以便计算损失
            targets = targets.view(B*T) # 将targets展平为 (B*T,)的形状，以便计算损失
            loss = F.cross_entropy(logits, targets) # 计算交叉熵损失

        return logits, loss # 返回logits和损失

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 为什么可以直接self(idx)?
            # 因为在PyTorch中，当我们调用模型实例时，会自动调用forward方法。
            # 这里的idx是当前的输入序列，形状是 (batch_size, block_size)
            logits, loss = self(idx, None) # 前向传播，计算输出
            logits = logits[:, -1, :] # 取出最后一个时间步的logits，形状是 (batch_size, vocab_size)

            # 为什么要用F.softmax(logits, dim=-1)呢？
            # 因为我们希望将logits转换为概率分布，以便从中采样下一个token的索引。
            # F.softmax函数会对logits进行归一化，使得每个类别的概率和为1。
            probs = F.softmax(logits, dim=-1) # 计算softmax概率, 形状是 (batch_size, vocab_size)
            # 为什么要用torch.multinomial(probs, num_samples=1)呢？
            # 因为我们希望从概率分布中采样下一个token的索引。
            # torch.multinomial函数会根据给定的概率分布，从中采样num_samples个样本。
            idx_next = torch.multinomial(probs, num_samples=1) # 选择概率最高的token作为下一个token，形状是 (batch_size, 1)
            idx = torch.cat((idx, idx_next), dim=1) # 将采样得到的下一个token的索引拼接到输入序列的末尾，形状是 (batch_size, block_size+1)
        return idx # 返回生成的序列

        
m = BigramLanguageModel(vocab_size) # 创建一个大ram语言模型实例
# 为什么要用m(xb, yb)呢？
# 因为我们需要将输入的tokens映射到logits，然后再计算损失。
# 这里的xb和yb是输入的tokens和目标tokens。
# 那为什么没有显式写forwards?
# 因为在PyTorch中，当我们调用模型实例时，会自动调用forward方法。
logits, loss = m(xb, yb) # 前向传播，计算输出
print(logits.shape) # 输出形状应该是 (batch_size, block_size, vocab_size)
print(loss) # 输出损失

# 为什么要用torch.zeros((1, 1), dtype=torch.long)呢？
# 因为我们需要一个初始的输入序列，形状是 (batch_size, block_size)
# 这里的batch_size是1，因为我们只需要生成一个序列。
# 这里的block_size是1，因为我们只需要生成一个token。

# 为什么要用m.generate(idx, max_new_tokens=100)呢？
# 这里的torch.zeros((1,1), dtype=torch.long)是初始的输入序列，形状是 (batch_size, block_size)
# 这里的max_new_tokens是生成序列的最大长度。
# 为什么要用[0].tolist()呢？
# 去除batch维度，得到一个一维的索引列表。
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=10)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWkt


In [ ]:
# m.parameters()返回模型的所有参数，optimizer.step()会根据计算得到的梯度更新这些参数。
# 更新的参数有什么?
# 1. 权重参数
# 2. 偏置参数
# 3. 学习率
# 4. 动量参数
# 5. 正则化参数
# 6. 梯度裁剪参数
# 7. 权重衰减参数
# 8. 学习率衰减参数
# 9. 学习率偏置参数
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) # 创建一个AdamW优化器，学习率为1e-3

In [47]:
batch_size = 32 # 每个批次的样本数量

for steps in range(10000):
    xb, yb = get_batch('train_data') # 获取一个训练批次
    logits, loss = m(xb, yb) # 前向传播，计算输出
    optimizer.zero_grad(set_to_none=True) # 清空梯度
    loss.backward() # 反向传播，计算梯度
    optimizer.step() # 更新参数

    print(f"step {steps}: loss {loss.item()}") # 输出当前步数和损失

step 0: loss 3.7509355545043945
step 1: loss 3.6942591667175293
step 2: loss 3.6222007274627686
step 3: loss 3.699336051940918
step 4: loss 3.5494465827941895
step 5: loss 3.6492812633514404
step 6: loss 3.6620051860809326
step 7: loss 3.562589645385742
step 8: loss 3.7036004066467285
step 9: loss 3.6513171195983887
step 10: loss 3.72029972076416
step 11: loss 3.5407874584198
step 12: loss 3.675413131713867
step 13: loss 3.6038851737976074
step 14: loss 3.6191442012786865
step 15: loss 3.644010305404663
step 16: loss 3.7355549335479736
step 17: loss 3.6427009105682373
step 18: loss 3.5594482421875
step 19: loss 3.705571413040161
step 20: loss 3.6020286083221436
step 21: loss 3.702944278717041
step 22: loss 3.6254076957702637
step 23: loss 3.750091075897217
step 24: loss 3.588874101638794
step 25: loss 3.683772325515747
step 26: loss 3.6604552268981934
step 27: loss 3.678088426589966
step 28: loss 3.563584327697754
step 29: loss 3.6056361198425293
step 30: loss 3.660257339477539
step 31

In [50]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Th Rishondecharvyouk p IVIImere we keicetlot llly ide ply t com: chavily; o fill wf mes co it hitodntly d d wsue ater to ovithA herisch ave une, t e pothamef. g st ofa nde hewhilero f tu dreanket iser oust he, s
s t,end ald atant

Turce dinsest; fowis MI ar gd yomblthaghrenenke this imy'dious wen histute ait MNow.
Shid thod thokedanenj's.
DUK: ishouthat,
's'd?
S:'lENoran igoventitean he, tt usWhyour ixxff ou blvelife eare mie bu pamak.
COnithololodaketwer thait g ESoco pe teran Gly? go!
Slirenkn


# self-attention中的数学技巧

In [ ]:
# 一个例子
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

B, T, C = 4, 8, 2 # B是batch_size，T是block_size，C是vocab_size
x = torch.randn(B, T, C) # 随机生成一个形状为 (B, T, C) 的张量
print(x.shape) # 输出张量的形状

torch.Size([4, 8, 2])


In [ ]:
xbow = torch.zeros((B, T, C)) # 创建一个形状为 (B, T, C) 的零张量，用于存储每个样本的词袋表示
for b in range(B): # 遍历每个样本
    for t in range(T): # 遍历每个时间步
        xprev = x[b, :t+1] # 获取第b个样本在第t个时间步的向量表示
        xbow[b, t] = torch.mean(xprev, dim=0) # 将前t+1个时间步的向量表示求和，得到词袋表示

In [ ]:
wei = torch.tril(torch.ones(T, T)) # 创建一个下三角矩阵，形状为 (T, T)
wei = wei / wei.sum(1, keepdim=True) # 将每一行归一化，使得每一行的和为1
# 为什么wei和x两个维数不同的矩阵能做乘法?
# wei的形状是 (T, T)，x的形状是 (B, T, C)，在进行矩阵乘法时，PyTorch会自动广播wei，使其与x的形状兼容。
xbow2 = wei @ x # 使用矩阵乘法计算词袋表示，形状为 (B, T, C)
torch.allclose(xbow, xbow2) # 检查两种方法计算的词袋表示是否相等

False

In [70]:
# version 3
tril = torch.tril(torch.ones(T, T)) # 创建一个下三角矩阵，形状为 (T, T)
wei = torch.zeros((T, T)) # 创建一个形状为 (T, T) 的零张量，用于存储权重矩阵
wei = wei.masked_fill(tril == 0, float('-inf')) # 将上三角部分的元素设置为负无穷大
wei = F.softmax(wei, dim=-1) # 对权重矩阵进行softmax归一化，使得每一行的和为1
xbow3 = wei @ x # 使用矩阵乘法计算词袋表示，形状为 (B, T, C)
torch.allclose(xbow2, xbow3) # 检查两种方法计算的词袋表示是否相等

True

In [80]:
# version 4
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C) # 随机生成一个形状为 (B, T, C) 的张量

head_size = 16 # 定义注意力头的大小
key = nn.Linear(C, head_size, bias=False) # 定义一个线性层，将输入的维度C映射到head_size
query = nn.Linear(C, head_size, bias=False) # 定义一个线性层，将输入的维度C映射到head_size
value = nn.Linear(C, head_size, bias=False) # 定义一个线性层，将输入的维度C映射到head_size

k = key(x) # 计算键向量，形状为 (B, T, head_size)
q = query(x) # 计算查询向量，形状为 (B, T, head_size)
v = value(x) # 计算值向量，形状为 (B, T, head_size)

wei = q @ k.transpose(-2, -1) # 计算注意力权重，形状为 (B, T, T)
tril = torch.tril(torch.ones(T, T)) # 创建一个下三角矩阵，形状为 (T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # 应用掩码，填充上三角部分为负无穷
wei = F.softmax(wei, dim=-1) # 应用softmax函数，进行归一化
out = wei @ v # 计算输出，形状为 (B, T, head_size)
out.shape # 输出形状应该是 (B, T, head_size)

torch.Size([4, 8, 16])

In [78]:
wei[0]

tensor([[-1.7629, -1.3011,  0.5652,  2.1616, -1.0674,  1.9632,  1.0765, -0.4530],
        [-3.3334, -1.6556,  0.1040,  3.3782, -2.1825,  1.0415, -0.0557,  0.2927],
        [-1.0226, -1.2606,  0.0762, -0.3813, -0.9843, -1.4303,  0.0749, -0.9547],
        [ 0.7836, -0.8014, -0.3368, -0.8496, -0.5602, -1.1701, -1.2927, -1.0260],
        [-1.2566,  0.0187, -0.7880, -1.3204,  2.0363,  0.8638,  0.3719,  0.9258],
        [-0.3126,  2.4152, -0.1106, -0.9931,  3.3449, -2.5229,  1.4187,  1.2196],
        [ 1.0876,  1.9652, -0.2621, -0.3158,  0.6091,  1.2616, -0.5484,  0.8048],
        [-1.8044, -0.4126, -0.8306,  0.5899, -0.7987, -0.5856,  0.6433,  0.6303]],
       grad_fn=<SelectBackward0>)